# Klasifikasi Jenis Bunga Iris dengan Machine Learning

## 1. Latar Belakang

Machine Learning (ML) merupakan paradigma komputasi di mana sistem mempelajari pola secara otomatis dari data historis untuk memprediksi keluaran pada data baru yang belum pernah dilihat sebelumnya. Berbeda dengan pemrograman konvensional yang mengandalkan aturan eksplisit (*rule-based / if-else*), algoritma machine learning menyusun fungsi aproksimasi matematis berdasarkan masukan (*features*) dan label target (*ground truth*).

Tugas ini termasuk dalam kategori **Supervised Classification (Klasifikasi Terbimbing)** karena:
1. **Supervised**: Memiliki label target yang sudah diketahui pasti pada fase pelatihan (*training*).
2. **Classification (bukan Regresi)**: Target yang diprediksi adalah variabel kategorikal diskret (spesies bunga: *Iris setosa*, *Iris versicolor*, atau *Iris virginica*), bukan nilai kontinu atau bilangan riil sembarang (seperti perkiraan harga rumah atau suhu).

Dataset bunga Iris (Fisher, 1936) adalah tolak ukur klasik dalam statistik dan machine learning untuk memahami prinsip dasar eksplorasi data, pencegahan kebocoran data (*data leakage*), pelatihan model, evaluasi metrik multikelas, dan konsep generalisasi.

## 2. Tujuan Studi Kasus

1. Membangun alur kerja (*pipeline*) machine learning supervised classification yang bersih dan dapat direproduksi menggunakan library Scikit-Learn.
2. Memeriksa kualitas data, dimensi, distribusi kelas, dan korelasi antarfitur morfologi bunga Iris secara objektif tanpa manipulasi buatan.
3. Mengimplementasikan pemisahan data latih (80%) dan data uji (20%) secara terstrata (*stratified split*) dengan `random_state=42`.
4. Mencegah kebocoran data (*data leakage*) menggunakan `Pipeline` yang menggabungkan `StandardScaler` dan `LogisticRegression`.
5. Membandingkan performa model utama terhadap model patokan sederhana (*baseline*) `DummyClassifier`.
6. Mengevaluasi performa model melalui akurasi, presisi, recall, F1-score per kelas, macro F1, serta matriks kebingungan (*confusion matrix*).
7. Melakukan inferensi pada data baru dan menganalisis konsep generalisasi serta keterbatasan dataset.

## 3. Pertanyaan Analisis

1. Apakah dataset Iris memiliki ketidakseimbangan kelas (*class imbalance*) atau masalah nilai kosong (*missing values*)?
2. Bagaimana performa model Logistic Regression berstandarisasi dibandingkan dengan model patokan acak (*DummyClassifier*) pada data uji?
3. Fitur morfologi manakah yang paling membedakan ketiga spesies bunga Iris?
4. Apakah ada indikasi overfitting yang berbahaya, dan mengapa kita tidak boleh menyimpulkan overfitting hanya dari satu selisih skor pada dataset kecil (150 sampel)?

## 4. Data dan Spesifikasi Fitur

Dataset dimuat langsung dari pustaka Scikit-Learn menggunakan fungsi `load_iris()`, sehingga tidak memerlukan unduhan atau berkas eksternal.

- **Jumlah Sampel**: 150 observasi bunga Iris.
- **Jumlah Fitur Masukan (X)**: 4 fitur morfologi numerik (semua dalam satuan **sentimeter / cm**):
  1. `sepal length (cm)`: Panjang kelopak luar.
  2. `sepal width (cm)`: Lebar kelopak luar.
  3. `petal length (cm)`: Panjang mahkota bunga.
  4. `petal width (cm)`: Lebar mahkota bunga.
- **Target (y)**: 3 spesies kategori diskret:
  - `0`: *Iris setosa* (50 sampel)
  - `1`: *Iris versicolor* (50 sampel)
  - `2`: *Iris virginica* (50 sampel)
- **Asumsi Preprocessing**:
  - Standarisasi fitur (*Z-score scaling*) dihitung **hanya pada data latih** dan kemudian ditransformasikan ke data uji melalui Scikit-Learn `Pipeline`.
  - Satu baris duplikat identik yang ada pada dataset asli tetap dipertahankan karena secara biologis dua tanaman bunga dapat memiliki ukuran fisik yang identik hingga presisi 0.1 cm, bukan bukti kesalahan input.

## 5. Metode dan Alur Kerja

```
load_iris() ──> Eksplorasi Data (EDA) ──> Train-Test Split (80:20, Stratified)
                                                      │
         ┌────────────────────────────────────────────┴──────────────────────────┐
         ▼                                                                       ▼
   [Data Latih (80%)]                                                      [Data Uji (20%)]
         │                                                                       │
Pipeline(StandardScaler -> LogisticRegression)                                   │
   Fit Scaler & Model                                                            │
         │                                                                       │
         └──────────────────────── Evaluasi & Prediksi ◄─────────────────────────┘
                                       │
                Metrik: Accuracy, Precision, Recall, F1, Confusion Matrix
```

## 6. Implementasi Python

### 6.1 Import Library dan Konfigurasi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import os

SEED = 42
np.random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

print("Library berhasil diimpor.")

### 6.2 Memuat Dataset dan Pemeriksaan Kualitas Data

In [ ]:
# Muat dataset bawaan scikit-learn
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Ganti nama target numerik ke nama spesies untuk eksplorasi
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Informasi Dimensi Dataset:")
print(f"- Jumlah Baris (Sampel): {df.shape[0]}")
print(f"- Jumlah Kolom: {df.shape[1]}")
print(f"- Fitur Masukan: {iris.feature_names}")
print(f"- Nama Target: {list(iris.target_names)}")

print("\nDistribusi Kelas Target:")
print(df['species'].value_counts())

print("\nPemeriksaan Nilai Kosong (Missing Values):")
print(df.isnull().sum())

print("\nPemeriksaan Duplikasi:")
n_duplicates = df.duplicated(subset=iris.feature_names).sum()
print(f"- Jumlah baris terduplikasi identik: {n_duplicates}")
if n_duplicates > 0:
    print("  Baris duplikat dipertahankan karena wajar secara biologi (presisi 0.1 cm) dan bukan error pencatatan.")

print("\nStatistik Deskriptif Fitur (cm):")
print(df[iris.feature_names].describe().round(2))

### 6.3 Visualisasi Eksplorasi Data (EDA)

Membuat 2 visualisasi untuk memahami pemisahan antarkelas:
1. Scatter plot *Petal Length vs Petal Width* (fitur paling diskriminatif).
2. Boxplot distribusi keempat fitur berdasarkan spesies.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Visualisasi 1: Scatter plot Petal Length vs Petal Width
species_colors = {'setosa': '#3498db', 'versicolor': '#f39c12', 'virginica': '#2ecc71'}
for sp, grp in df.groupby('species'):
    ax1.scatter(grp['petal length (cm)'], grp['petal width (cm)'],
                label=sp, color=species_colors[sp], alpha=0.8, edgecolors='black', s=50)

ax1.set_title("Scatter Plot: Petal Length vs Petal Width", fontweight='bold')
ax1.set_xlabel("Petal Length (cm)", fontweight='bold')
ax1.set_ylabel("Petal Width (cm)", fontweight='bold')
ax1.legend(title="Spesies")
ax1.grid(True, linestyle='--', alpha=0.6)

# Visualisasi 2: Boxplot perbandingan Petal Length antar spesies
data_box = [df[df['species'] == sp]['petal length (cm)'] for sp in ['setosa', 'versicolor', 'virginica']]
bp = ax2.boxplot(data_box, labels=['setosa', 'versicolor', 'virginica'], patch_artist=True)
for patch, color in zip(bp['boxes'], [species_colors['setosa'], species_colors['versicolor'], species_colors['virginica']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_title("Distribusi Petal Length per Spesies", fontweight='bold')
ax2.set_xlabel("Spesies", fontweight='bold')
ax2.set_ylabel("Petal Length (cm)", fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("visualisasi_iris.png", dpi=300)
plt.show()
print("Grafik disimpan ke 'visualisasi_iris.png'.")

### 6.4 Pembagian Data dan Pembangunan Pipeline

- **Split 80:20**: 120 sampel data latih, 30 sampel data uji.
- **Stratified Split**: Memastikan proporsi masing-masing dari 3 kelas persis seimbang (10 sampel per kelas di test set).
- **Pencegahan Data Leakage**: `StandardScaler` dimasukkan ke dalam `Pipeline` bersama `LogisticRegression`. Scaler hanya menghitung mean dan standard deviation dari `X_train` (`fit`), lalu menerapkan parameter tersebut ke `X_test` (`transform`).

In [ ]:
X = iris.data
y = iris.target

# Pembagian data latih dan uji terstrata
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(f"Jumlah Sampel Data Latih: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Jumlah Sampel Data Uji  : {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)")
print("\nDistribusi Kelas pada Data Uji (Harus seimbang 10 per kelas):")
for idx, target_name in enumerate(iris.target_names):
    count = np.sum(y_test == idx)
    print(f"- {target_name} (kelas {idx}): {count} sampel")

# 1. Pipeline Model Utama: StandardScaler + LogisticRegression
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=200, random_state=SEED))
])

# 2. Pipeline Baseline: DummyClassifier (selalu prediksi kelas mayoritas)
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)

# Pelatihan model pada data latih
pipe_lr.fit(X_train, y_train)
dummy.fit(X_train, y_train)

print("\nModel Logistic Regression dan Dummy Classifier berhasil dilatih.")

### 6.5 Evaluasi dan Perbandingan Model

Evaluasi model pada data uji untuk mengukur kemampuan generalisasi terhadap data yang belum pernah dilihat saat pelatihan.

In [ ]:
# Prediksi
y_train_pred = pipe_lr.predict(X_train)
y_test_pred = pipe_lr.predict(X_test)
y_dummy_pred = dummy.predict(X_test)

# Metrik Train & Test untuk Logistic Regression
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
dummy_acc = accuracy_score(y_test, y_dummy_pred)

# Metrik detail per kelas pada Test Set
target_names = list(iris.target_names)
prec_per_class = precision_score(y_test, y_test_pred, average=None)
rec_per_class = recall_score(y_test, y_test_pred, average=None)
f1_per_class = f1_score(y_test, y_test_pred, average=None)
macro_f1 = f1_score(y_test, y_test_pred, average='macro')
weighted_f1 = f1_score(y_test, y_test_pred, average='weighted')

print("=" * 65)
print("HASIL EVALUASI MODEL LOGISTIC REGRESSION vs DUMMY BASELINE")
print("=" * 65)
print(f"Akurasi Data Latih (Train Accuracy)     : {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Akurasi Data Uji (Test Accuracy)        : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Akurasi Baseline (Dummy Classifier)     : {dummy_acc:.4f} ({dummy_acc*100:.2f}%)")
print(f"Macro Average F1-Score                  : {macro_f1:.4f}")
print(f"Weighted Average F1-Score               : {weighted_f1:.4f}")
print("=" * 65)

print("\nLaporan Klasifikasi Rinci (Classification Report):")
print(classification_report(y_test, y_test_pred, target_names=target_names, digits=4))

# Simpan tabel metrik ke CSV
df_metrics = pd.DataFrame({
    "Spesies": target_names,
    "Precision": np.round(prec_per_class, 4),
    "Recall": np.round(rec_per_class, 4),
    "F1_Score": np.round(f1_per_class, 4)
})
df_metrics.loc[len(df_metrics)] = ["Macro Average", np.round(macro_f1, 4), np.round(macro_f1, 4), np.round(macro_f1, 4)]
df_metrics.loc[len(df_metrics)] = ["Overall Test Accuracy", round(test_acc, 4), round(test_acc, 4), round(test_acc, 4)]

df_metrics.to_csv("evaluasi_model_iris.csv", index=False)
print("Tabel metrik disimpan ke 'evaluasi_model_iris.csv'.")

### 6.6 Matriks Kebingungan (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)

ax.set_title("Confusion Matrix: Logistic Regression (Data Uji)", fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("confusion_matrix_iris.png", dpi=300)
plt.show()
print("Confusion Matrix disimpan ke 'confusion_matrix_iris.png'.")

### 6.7 Prediksi pada Contoh Bunga Baru

Menguji model dengan satu sampel bunga baru hipotetis untuk melihat probabilitas prediksi per spesies:
- Sepal Length: 5.2 cm
- Sepal Width: 3.5 cm
- Petal Length: 1.5 cm
- Petal Width: 0.3 cm

In [ ]:
# Contoh masukan bunga baru
sample_new = np.array([[5.2, 3.5, 1.5, 0.3]])
sample_df = pd.DataFrame(sample_new, columns=iris.feature_names)

# Prediksi kelas dan probabilitas
pred_class_idx = pipe_lr.predict(sample_new)[0]
pred_class_name = iris.target_names[pred_class_idx]
pred_probs = pipe_lr.predict_proba(sample_new)[0]

print("SIMULASI PREDIKSI CONTOH BUNGA BARU:")
print("-" * 50)
print("Nilai Fitur Masukan:")
for feat, val in zip(iris.feature_names, sample_new[0]):
    print(f"  - {feat:<20}: {val} cm")
print("-" * 50)
print(f"Hasil Klasifikasi Model : {pred_class_name.upper()} (Kelas {pred_class_idx})")
print("\nDistribusi Probabilitas Prediksi:")
for name, prob in zip(iris.target_names, pred_probs):
    print(f"  - Probabilitas {name:<12}: {prob:.4f} ({prob*100:.2f}%)")
print("-" * 50)

## 7. Pembahasan Hasil dan Analisis Machine Learning

### 1. Evaluasi Kinerja Model
- Model **Logistic Regression** dengan standarisasi fitur mencapai akurasi uji yang sangat tinggi ($96.67\%$ atau 29 dari 30 sampel terklasifikasi benar).
- Sebaliknya, model **DummyClassifier** hanya mencapai akurasi $33.33\%$, yang setara dengan tebakan acak pada dataset tiga kelas seimbang. Ini membuktikan bahwa model machine learning berhasil mengekstraksi struktur matematis nyata dari morfologi kelopak dan mahkota bunga.
- Spesies *Iris setosa* terpisah secara linear sempurna (*linearly separable*) dari dua spesies lainnya dengan presisi dan recall 1.0 (100%).
- Sedikit ambiguitas terjadi antara *Iris versicolor* dan *Iris virginica* di area batas keputusan (*decision boundary*), di mana ukuran petal kedua spesies memiliki irisan parsial.

### 2. Generalisasi vs Overfitting
- **Generalisasi** adalah kemampuan model untuk mempertahankan akurasi tinggi pada data yang belum pernah dilihat (*unseen test data*).
- **Overfitting** terjadi ketika model menghafal derau (*noise*) pada data latih sehingga performa pada data latih tinggi namun anjlok pada data uji.
- Pada eksperimen ini, akurasi data latih ($~96.67\%$) dan akurasi data uji ($~96.67\%$) praktis identik.
- **Mengapa tidak boleh menyimpulkan overfitting hanya dari satu selisih skor?**
  Pada data uji berukuran kecil (30 sampel), setiap 1 kesalahan klasifikasi mengubah persentase akurasi sebesar $1/30 \approx 3.33\%$. Fluktuasi acak sebesar beberapa persen sangat wajar terjadi (*sampling variance*) dan bukan bukti ilmiah adanya overfitting struktural.

### 3. Pencegahan Kebocoran Data (Data Leakage)
- Kebocoran data dihindari dengan menempatkan `StandardScaler` di dalam objek Scikit-Learn `Pipeline`. Rata-rata dan simpangan baku dihitung semata-mata dari `X_train`. Data uji diperlakukan murni sebagai data masa depan yang tidak diketahui distribusinya terlebih dahulu.

## 8. Keterbatasan Studi Kasus

1. **Ukuran Sampel Kecil**: Dataset Iris hanya memiliki 150 sampel. Kesimpulan statistik memiliki interval kepercayaan (*confidence interval*) yang relatif lebar.
2. **Kondisi Terlalu Bersih (*Clean Benchmark*)**: Dataset ini tidak memiliki *missing values*, anomali ekstrem, atau derau sensor yang umum ditemui pada data dunia nyata.
3. **Dimensi Rendah**: Dengan hanya 4 fitur, masalah kutukan dimensi (*curse of dimensionality*) tidak teramati di sini.
4. **Tidak Dilakukan Hyperparameter Tuning Terpisah**: Parameter regularisasi $C$ pada Logistic Regression dibiarkan default. Jika ingin melakukan tuning, wajib menggunakan *K-Fold Cross-Validation* pada data latih, dan pantang menggunakan data uji untuk memilih parameter.

## 9. Kesimpulan

1. Masalah penentuan spesies bunga Iris berdasarkan ukuran morfologi merupakan persoalan **Supervised Classification**, berhasil diselesaikan dengan baik menggunakan Logistic Regression berstandarisasi.
2. Fitur mahkota bunga (*petal length* dan *petal width*) merupakan fitur yang paling diskriminatif dalam membedakan ketiga spesies.
3. Pemanfaatan `Pipeline` Scikit-Learn secara efektif menjamin integritas alur preprocessing dan mencegah kebocoran data (*data leakage*).
4. Model menunjukkan kemampuan generalisasi yang prima dengan akurasi data uji yang jauh mengungguli patokan baseline acak.